In [ ]:
import os

import pandas as pd
from src import analysis, common

os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Use CPU for evaluation

In [ ]:
# ======================================================================
# Configuration
# ======================================================================
TASK = "steady_flow"
RUN_NAME = "replace_with_current_run_name"
DATASET_ROOT = common.paths.get_dataset_root()

RUN_DIR = common.paths.resolve_run_output_dir(TASK, RUN_NAME)
plan = analysis.artifact_service.load_run_artifact_plan(RUN_DIR)

artifact_splits = {
    plan.id_dataset_name: "eval",
    plan.ood_dataset_name: "ood",
}

In [ ]:
def run_or_load_artifacts_evaluation(*, dataset_name: str, split: str) -> pd.DataFrame:
    """Load or generate artifacts through the split-aware artifact service."""
    return analysis.artifact_service.run_or_load_artifacts(
        run_dir=RUN_DIR,
        dataset_name=dataset_name,
        split=split,
        max_cases=None,
        batch_size=1,
        prefer_cuda=False,
        dataset_root=DATASET_ROOT,
        rebuild=False,
    )

In [ ]:
# ---------------------------------------------------------------------
# Generate or load artifacts, then build evaluation dataframes
# ---------------------------------------------------------------------
datasets_raw = {}
datasets_eval = {}

for dataset_name, split in artifact_splits.items():
    df_raw = run_or_load_artifacts_evaluation(dataset_name=dataset_name, split=split)
    if df_raw.empty:
        print(f"[WARN] No artifacts available for {dataset_name} ({split})")
        continue
    datasets_raw[dataset_name] = df_raw
    datasets_eval[dataset_name] = analysis.evaluation.dataframe.build_eval_df(df_raw)

In [ ]:
panel = analysis.evaluation.panel.build_evaluation_panel(
    datasets_eval=datasets_eval,
    title=RUN_NAME,
    sections=[
        "global_error",
        "error_decomposition",
        "physical_consistency",
        "spectral",
        "error_sensitivity",
        "sample_viewer",
        "outliers",
    ],
)

display(panel)